# Technology Diffusion Results

Analysis of the main technology-diffusion experiment results, based on the original `results_try.ipynb` workflow.


## Imports and Plot Configuration


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
import pandas as pd
import ast
import json
import numpy as np
import seaborn as sns


plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "figure.titlesize": 18,
})

ALGO_COLORS = {
    "coads_binary_search": "#d62728",  # most important
    "approx": "#1f77b4",              # most important
    "goldberg_liu": "#2ca02c",        # most important
    "degree_threshold": "#9467bd",
    "degree_discount": "#ff7f0e",
    "degree_connected": "#8c564b",
    "degree": "#e377c2",
    "betweenness": "#17becf",
    "high_thetas": "#7f7f7f",
}

ALGO_LABELS = {
    "coads_binary_search": "COaDS Binary Search",
    "approx": "Approximation",
    "goldberg_liu": "Goldberg–Liu",
    "degree_threshold": "Degree Threshold",
    "degree_discount": "Degree Discount",
    "degree_connected": "Degree Connected",
    "degree": "Degree",
    "betweenness": "Betweenness",
    "high_thetas": "High Thetas",
}

ALGO_LINESTYLES = {
    "coads_binary_search": "-",
    "approx": "--",
    "goldberg_liu": "-.",
    "degree_threshold": ":",
    "degree_discount": (0, (5, 1)),
    "degree_connected": (0, (3, 1, 1, 1)),
    "degree": (0, (2, 1)),
    "betweenness": (0, (1, 1)),
    "high_thetas": (0, (4, 2)),
}


def algo_label(name):
    if name is None:
        return "Unknown"
    name = str(name)
    return ALGO_LABELS.get(name, name.replace("_", " ").title())


def format_instance_tick(instance):
    n, c = instance
    if float(n).is_integer():
        n_txt = f"{int(n):,}"
    else:
        n_txt = f"{n:g}"

    if float(c).is_integer():
        c_txt = f"{int(c)}"
    else:
        c_txt = f"{c:g}"

    return f"({n_txt},{c_txt})"

save = 1

project_root = Path.cwd()
if not (project_root / "results").exists() and (project_root.parent / "results").exists():
    project_root = project_root.parent
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

## Load and Normalize Results


In [ ]:
project_root = Path.cwd()
if not (project_root / "results").exists() and (project_root.parent / "results").exists():
    project_root = project_root.parent

results_dir = project_root / "results" / "technology diffusion"
if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir}")

temp_results_dir = project_root / "results" / "temp"

def normalize_result_frame(frame, source_file):
    frame = frame.copy()
    frame["source_file"] = source_file.name

    if "history" not in frame.columns:
        frame["history"] = None
    if "selected_seed_nodes" not in frame.columns:
        frame["selected_seed_nodes"] = None

    def parse_selected_seed_nodes(raw):
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            return None
        if isinstance(raw, str):
            raw = raw.strip()
            if not raw:
                return None
            try:
                return json.loads(raw)
            except json.JSONDecodeError:
                return ast.literal_eval(raw)
        if isinstance(raw, (list, tuple)):
            return list(raw)
        return raw

    frame["selected_seed_nodes"] = frame["selected_seed_nodes"].apply(parse_selected_seed_nodes)
    return frame

result_csv_files = sorted(results_dir.glob("technology_diffusion_results__*.csv"))
if not result_csv_files:
    fallback = results_dir / "technology_diffusion_results.csv"
    if fallback.exists():
        result_csv_files = [fallback]

frames = [normalize_result_frame(pd.read_csv(file), file) for file in result_csv_files]

if not frames:
    raise FileNotFoundError("No technology_diffusion_results CSV files were found.")

df_results = pd.concat(frames, ignore_index=True)

for col in ["n_nodes", "c", "seed", "total_time", "K"]:
    if col in df_results.columns:
        df_results[col] = pd.to_numeric(df_results[col], errors="coerce")

# Worst-case fallback: if K is missing, set K=n_nodes and inject history=[[n_nodes, 0.0]].
if "K" in df_results.columns and "n_nodes" in df_results.columns:
    missing_k_mask = df_results["K"].isna() & df_results["n_nodes"].notna()
    df_results.loc[missing_k_mask, "K"] = df_results.loc[missing_k_mask, "n_nodes"].astype(float)
    if "history" in df_results.columns:
        df_results.loc[missing_k_mask, "history"] = df_results.loc[missing_k_mask, "n_nodes"].apply(
            lambda n: json.dumps([[float(n), 0.0]])
        )

if "algorithm" in df_results.columns and "history" in df_results.columns and "n_nodes" in df_results.columns:
    history_str = df_results["history"].astype("string")
    approx_mask = df_results["algorithm"].eq("approx") & df_results["n_nodes"].notna()

    approx_missing_history = approx_mask & (
        df_results["history"].isna() | history_str.str.strip().isin(["", "[]"])
    )
    df_results.loc[approx_missing_history, "history"] = df_results.loc[approx_missing_history, "n_nodes"].apply(
        lambda n: json.dumps([[float(n), 0.0]])
    )

    def ensure_approx_start_event(raw, n):
        n = float(n)
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            return json.dumps([[n, 0.0]])

        parsed = raw
        if isinstance(raw, str):
            raw = raw.strip()
            if not raw:
                return json.dumps([[n, 0.0]])
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                parsed = ast.literal_eval(raw)

        if not isinstance(parsed, (list, tuple)) or len(parsed) == 0:
            return json.dumps([[n, 0.0]])

        clean = []
        for point in parsed:
            if not isinstance(point, (list, tuple)) or len(point) < 2:
                continue
            try:
                k = float(point[0])
                t = float(point[1])
            except (TypeError, ValueError):
                continue
            if np.isfinite(k) and np.isfinite(t):
                clean.append([k, t])

        if not clean:
            return json.dumps([[n, 0.0]])

        clean.sort(key=lambda p: p[1])
        has_start = any(np.isclose(t, 0.0, atol=1e-12) and np.isclose(k, n, atol=1e-9) for k, t in clean)
        if not has_start:
            clean = [[n, 0.0]] + clean

        return json.dumps(clean)

    df_results.loc[approx_mask, "history"] = df_results.loc[approx_mask].apply(
        lambda row: ensure_approx_start_event(row.get("history"), row.get("n_nodes")),
        axis=1,
    )

print(
    f"Loaded {len(result_csv_files)} result files and {len(df_results)} total rows from {results_dir}."
)
df_results.head()

## Mean K by Number of Nodes


In [ ]:
algorithm_order = [
    "degree_threshold",
    "degree_discount",
    "degree_connected",
    "degree",
    "betweenness",
    "approx",
    "goldberg_liu",
    "coads_binary_search",
]

plot_df = (
    df_results.groupby(["n_nodes", "algorithm"], as_index=False)
    .agg(mean_K=("K", "mean"), mean_total_time=("total_time", "mean"))
    .sort_values(["n_nodes", "algorithm"])
)

pivot_k = plot_df.pivot(index="n_nodes", columns="algorithm", values="mean_K").sort_index()
available_algorithms = [a for a in algorithm_order if a in pivot_k.columns]

if not available_algorithms:
    raise ValueError("No expected algorithms found in df_results.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), dpi=120)

markers = ["o", "s", "^", "D", "P", "X", "v", "<"]
fallback_gl = pd.Series(pivot_k.index, index=pivot_k.index, dtype=float)

for i, alg in enumerate(available_algorithms):
    series = pivot_k[alg]
    if alg == "goldberg_liu":
        series = series.fillna(fallback_gl)

    ax.plot(
        pivot_k.index,
        series,
        marker=markers[i % len(markers)],
        linewidth=1.6,
        linestyle=ALGO_LINESTYLES.get(alg, "-"),
        color=ALGO_COLORS.get(alg, "#4c4c4c"),
        label=algo_label(alg),
    )

ax.set_xlabel("Number of Nodes")
ax.set_ylabel("Mean B (log scale)")
ax.set_yscale("log")
ax.set_xticks(pivot_k.index.to_list())
ax.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=True, framealpha=0.9, ncols=2, loc="best")

fig.tight_layout()
if save:
    fig.savefig(figures_dir / "mean_k_by_nodes.png", dpi=220, bbox_inches="tight")
plt.show()

## Mean History Step-Curves


In [ ]:
show_std_band = 0

def parse_history(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return None
        try:
            raw = json.loads(raw)
        except json.JSONDecodeError:
            raw = ast.literal_eval(raw)

    if not isinstance(raw, (list, tuple)) or len(raw) == 0:
        return None

    arr = np.asarray(raw, dtype=float)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return None

    values = arr[:, 0]
    times = arr[:, 1]

    finite_mask = np.isfinite(values) & np.isfinite(times)
    if not np.any(finite_mask):
        return None

    values = values[finite_mask]
    times = times[finite_mask]

    order = np.argsort(times)
    return times[order], values[order]

def mean_step_curve(histories, t_end=None):
    parsed = [parse_history(h) for h in histories]
    parsed = [p for p in parsed if p is not None]
    if not parsed:
        return None

    all_event_times = np.concatenate([p[0] for p in parsed])
    all_event_times = all_event_times[np.isfinite(all_event_times)]
    if all_event_times.size == 0:
        return None

    if t_end is not None and float(t_end) > 0:
        t_end = float(t_end)
        all_event_times = all_event_times[all_event_times <= t_end]
        t_grid = np.unique(np.concatenate(([0.0], all_event_times, [t_end])))
    else:
        t_grid = np.unique(np.concatenate(([0.0], all_event_times)))

    mean_values = []
    for t_hist, k_hist in parsed:
        idx = np.searchsorted(t_hist, t_grid, side="right") - 1
        idx = np.clip(idx, 0, len(k_hist) - 1)
        mean_values.append(k_hist[idx])

    stacked = np.vstack(mean_values)
    return t_grid, stacked.mean(axis=0)

# Compatibility helper for downstream merged plotting cell.
def mean_std_curve(histories, t_end, n_points=300):
    curve = mean_step_curve(histories, t_end=t_end)
    if curve is None:
        return None
    t_grid, y_mean = curve
    y_std = np.zeros_like(y_mean)
    return t_grid, y_mean, y_std

def best_k_time_from_history(raw_history):
    parsed = parse_history(raw_history)
    if parsed is None:
        return None

    t_hist, k_hist = parsed
    if t_hist.size == 0 or k_hist.size == 0:
        return None

    best_idx = int(np.argmin(k_hist))
    best_k = float(k_hist[best_idx])

    same_best_mask = np.isclose(k_hist, best_k, rtol=0.0, atol=1e-12)
    best_times = t_hist[same_best_mask]
    if best_times.size == 0:
        return None

    best_time = float(np.min(best_times))
    return best_k, best_time

n_values = sorted(df_results["n_nodes"].dropna().unique())
if not n_values:
    raise ValueError("No n_nodes values found in df_results.")

history_algorithms = [a for a in available_algorithms if a != "high_thetas"]

for n in n_values:
    df_n = df_results[df_results["n_nodes"] == n].copy()
    budget_s = (n / 100) * 100
    fig, ax = plt.subplots(figsize=(11, 6), dpi=120)

    plotted = 0
    for alg in history_algorithms:
        histories = df_n.loc[df_n["algorithm"] == alg, "history"].tolist()
        curve = mean_step_curve(histories, t_end=budget_s)
        if curve is None:
            continue
        t_alg, y_mean = curve
        t_plot = np.maximum(t_alg, 1e-3)
        mask_budget = t_plot <= budget_s
        t_plot = t_plot[mask_budget]
        y_mean = y_mean[mask_budget]
        if t_plot.size == 0:
            continue
        ax.step(
            t_plot,
            y_mean,
            where="post",
            linewidth=1.6,
            linestyle=ALGO_LINESTYLES.get(alg, "-"),
            color=ALGO_COLORS.get(alg, "#4c4c4c"),
            label=algo_label(alg),
        )
        plotted += 1

    if plotted == 0:
        plt.close(fig)
        continue

    ax.axvline(
        budget_s,
        color="#d62728",
        linestyle="--",
        linewidth=1.2,
        alpha=0.9,
        label="Time budget",
    )
    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.set_xlim(1e-3, budget_s)
    ax.set_xlabel("Time (s, log scale)")
    ax.set_ylabel("Mean B")
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.65)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(loc="lower left", frameon=True, framealpha=0.9, ncols=2)

    fig.tight_layout()
    if save:
        fig.savefig(figures_dir / f"mean_history_n_{int(n)}.png", dpi=220, bbox_inches="tight")
    plt.show()

## Rank Distribution by Algorithm


In [ ]:
candidate_dims = ["n_nodes", "c", "seed"]
instance_keys = [col for col in candidate_dims if col in df_results.columns]

if "algorithm" not in df_results.columns or "K" not in df_results.columns:
    raise ValueError("df_results must contain at least 'algorithm' and 'K' columns.")
if "n_nodes" not in df_results.columns:
    raise ValueError("df_results must contain 'n_nodes' for this plot.")
if not instance_keys:
    raise ValueError("At least one of ['n_nodes', 'c', 'seed'] must be present in df_results.")

required_cols = instance_keys + ["algorithm"]
rank_source_first = df_results.dropna(subset=required_cols).copy()
rank_source_first = rank_source_first[rank_source_first["algorithm"] != "high_thetas"].copy()

rank_algorithms = [
    a for a in algorithm_order
    if a in rank_source_first["algorithm"].unique() and a != "high_thetas"
 ]
if not rank_algorithms:
    rank_algorithms = sorted(rank_source_first["algorithm"].dropna().unique())

instances = rank_source_first[instance_keys].drop_duplicates().copy()
instances["_k"] = 1
algorithms_df = pd.DataFrame({"algorithm": rank_algorithms, "_k": 1})

rank_complete = instances.merge(algorithms_df, on="_k", how="inner").drop(columns=["_k"])
k_values = rank_source_first[instance_keys + ["algorithm", "K"]].copy()
rank_complete = rank_complete.merge(k_values, on=instance_keys + ["algorithm"], how="left")

rank_complete["K_for_rank"] = rank_complete["K"].fillna(np.inf)
rank_complete["rank_in_instance"] = (
    rank_complete.groupby(instance_keys)["K_for_rank"]
    .rank(method="min", ascending=True)
    .astype(int)
)

for n in sorted(rank_complete["n_nodes"].dropna().unique()):
    df_n = rank_complete[rank_complete["n_nodes"] == n].copy()
    if df_n.empty:
        continue

    box_data = [
        df_n.loc[df_n["algorithm"] == alg, "rank_in_instance"].dropna().to_numpy()
        for alg in rank_algorithms
    ]

    if all(len(arr) == 0 for arr in box_data):
        continue

    fig, ax = plt.subplots(figsize=(11, 6), dpi=120)
    bp = ax.boxplot(
        box_data,
        tick_labels=[algo_label(alg) for alg in rank_algorithms],
        patch_artist=True,
        showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "#d62728", "markeredgecolor": "#d62728", "markersize": 5},
    )

    for patch in bp["boxes"]:
        patch.set_facecolor("#9ecae1")
        patch.set_alpha(0.75)
    for median in bp["medians"]:
        median.set_color("#1f77b4")
        median.set_linewidth(2.0)

    ax.set_xlabel("Algorithm")
    ax.set_ylabel("Rank")
    ax.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.xticks(rotation=25, ha="right")

    fig.tight_layout()
    if save:
        fig.savefig(figures_dir / f"rank_boxplot_n_{int(n)}.png", dpi=220, bbox_inches="tight")
    plt.show()

## Statistics


In [ ]:
# --- Global dataset statistics ---
print("=== Dataset statistics ===")
print(f"Rows: {len(df_results):,}")

if "algorithm" in df_results.columns:
    alg_values = sorted(df_results["algorithm"].dropna().unique())
    print(f"Algorithms: {len(alg_values)}")
    print("Algorithm names: " + ", ".join(str(a) for a in alg_values))

if "n_nodes" in df_results.columns:
    node_values = sorted(df_results["n_nodes"].dropna().unique())
    print(f"n_nodes values: {len(node_values)}")
    if node_values:
        node_text = ", ".join(str(int(v)) if float(v).is_integer() else f"{float(v):g}" for v in node_values)
        print("n_nodes set: " + node_text)
        print(f"n_nodes range: min={float(np.min(node_values)):g}, max={float(np.max(node_values)):g}")

if "seed" in df_results.columns:
    print(f"Unique seeds: {df_results['seed'].nunique()}")
if "c" in df_results.columns:
    print(f"Unique c values: {df_results['c'].nunique()}")

# --- Approx failure statistics: failure <=> K == n_nodes ---
if {"algorithm", "K", "n_nodes"}.issubset(df_results.columns):
    approx_df = df_results[df_results["algorithm"] == "approx"].copy()
    approx_df["K_num"] = pd.to_numeric(approx_df["K"], errors="coerce")
    approx_df["n_num"] = pd.to_numeric(approx_df["n_nodes"], errors="coerce")
    approx_df = approx_df.dropna(subset=["K_num", "n_num"]).copy()

    print("\n=== Approx failure statistics (K == n_nodes) ===")
    total_approx = len(approx_df)

    if total_approx:
        is_failure = np.isclose(approx_df["K_num"], approx_df["n_num"], rtol=0.0, atol=1e-12)
        failures = int(is_failure.sum())
        success = total_approx - failures
        fail_rate = failures / total_approx

        print(f"Total approx runs (with valid K and n_nodes): {total_approx:,}")
        print(f"Failures (K == n_nodes): {failures:,}")
        print(f"Successes: {success:,}")
        print(f"Failure rate: {fail_rate:.2%}")

        by_n = (
            approx_df.assign(is_failure=is_failure)
            .groupby("n_num", as_index=False)
            .agg(
                total_runs=("is_failure", "size"),
                failures=("is_failure", "sum"),
            )
        )
        by_n["failure_rate"] = by_n["failures"] / by_n["total_runs"]
        by_n = by_n.sort_values("n_num")
        by_n["n_nodes"] = by_n["n_num"].map(lambda x: int(x) if float(x).is_integer() else float(x))
        by_n = by_n[["n_nodes", "total_runs", "failures", "failure_rate"]]

        print("\nFailure breakdown by n_nodes:")
        print(
            by_n.to_string(
                index=False,
                formatters={
                    "failure_rate": "{:.2%}".format,
                },
            )
        )
    else:
        print("No valid approx rows with both K and n_nodes.")
else:
    print("\nMissing one of required columns: algorithm, K, n_nodes")

## Relative Gap Data


In [ ]:
required_cols = ["n_nodes", "c", "seed", "algorithm", "K"]
missing_cols = [col for col in required_cols if col not in df_results.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

gap_source = (
    df_results[required_cols]
    .dropna(subset=required_cols)
    .query("algorithm != 'high_thetas'")
    .copy()
)

gap_source = (
    gap_source.groupby(
        ["n_nodes", "c", "seed", "algorithm"],
        as_index=False,
        observed=True,
    )
    .agg(K=("K", "mean"))
)

gap_source["best_K"] = gap_source.groupby(
    ["n_nodes", "c", "seed"],
    observed=True,
 )["K"].transform("min")

denominator = gap_source["best_K"].abs()
gap_source["relative_gap"] = np.where(
    denominator > 1e-12,
    (gap_source["K"] - gap_source["best_K"]) / denominator,
    np.where(
        np.isclose(gap_source["K"], gap_source["best_K"]),
        0.0,
        np.nan,
    ),
)

mean_gap = (
    gap_source.groupby(
        ["n_nodes", "c", "algorithm"],
        observed=True,
    )["relative_gap"]
    .mean()
    .reset_index()
)

gap_heatmap = (
    mean_gap.pivot(
        index=["n_nodes", "c"],
        columns="algorithm",
        values="relative_gap",
    )
    .sort_index()
)

## Relative Gap Heatmap


In [ ]:
preferred_algorithm_order = [
    "degree_threshold",
    "degree_discount",
    "degree_connected",
    "degree",
    "betweenness",
    "approx",
    "goldberg_liu",
    "coads_binary_search",
]

gap_heatmap_pct = 100 * (
    mean_gap.pivot(
        index="algorithm",
        columns=["n_nodes", "c"],
        values="relative_gap",
    )
    .reindex(index=[
        alg for alg in preferred_algorithm_order
        if alg in mean_gap["algorithm"].unique()
    ])
    .sort_index(axis=1, level=[0, 1])
)

positive_values = gap_heatmap_pct.to_numpy()
positive_values = positive_values[
    np.isfinite(positive_values) & (positive_values > 0)
]

if positive_values.size == 0:
    raise ValueError("The heatmap contains no strictly positive relative gaps.")

color_floor = max(positive_values.min(), 1e-2)
color_ceiling = positive_values.max()

gap_heatmap_colors = gap_heatmap_pct.clip(lower=color_floor)

fig_width = max(11, 0.6 * gap_heatmap_pct.shape[1] + 3)
fig_height = max(6, 0.42 * gap_heatmap_pct.shape[0] + 1.5)

fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=120)
ax.set_facecolor("#fbfbfb")

sns.heatmap(
    gap_heatmap_colors,
    annot=gap_heatmap_pct,
    fmt=".2f",
    cmap="YlOrRd",
    norm=LogNorm(vmin=color_floor, vmax=color_ceiling),
    linewidths=0.6,
    linecolor="#f3f3f3",
    mask=gap_heatmap_pct.isna(),
    cbar_kws={"label": "Average relative gap (%)", "shrink": 0.92},
    annot_kws={"size": 9},
    ax=ax,
)

ax.set_xlabel("Instance (N, c)")
ax.set_ylabel("Algorithm")
ax.set_yticklabels([algo_label(idx) for idx in gap_heatmap_pct.index], rotation=0)
ax.set_xticklabels([format_instance_tick(idx) for idx in gap_heatmap_pct.columns], rotation=45, ha="right")
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", length=0)
fig.tight_layout()

if save:
    fig.savefig(
        figures_dir / "average_relative_gap_heatmap.png",
        dpi=220,
        bbox_inches="tight",
    )

plt.show()

## Result Preview


In [ ]:
df_results.head()

## Performance Profiles


In [ ]:
profile_metric = "K"
profile_agg = "mean"
tau_max_override = None
excluded_algorithms = {"high_thetas"}

required_columns = {"algorithm", "c", profile_metric}
missing_columns = required_columns.difference(df_results.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

instance_columns = [col for col in ["n_nodes", "c", "seed"] if col in df_results.columns]
if not instance_columns:
    raise ValueError("Expected at least one of ['n_nodes', 'c', 'seed'] in df_results.")

profile_source = df_results[instance_columns + ["algorithm", profile_metric]].copy()
profile_source[profile_metric] = pd.to_numeric(profile_source[profile_metric], errors="coerce")
profile_source = profile_source[~profile_source["algorithm"].isin(excluded_algorithms)].copy()

profile_algorithms = [
    alg for alg in algorithm_order
    if alg in profile_source["algorithm"].dropna().unique() and alg not in excluded_algorithms
]
profile_algorithms += sorted(
    set(profile_source["algorithm"].dropna().unique())
    .difference(profile_algorithms)
    .difference(excluded_algorithms)
)

if not profile_algorithms:
    raise ValueError("No algorithms are available for the performance profiles.")

c_values = sorted(profile_source["c"].dropna().unique())
if len(c_values) != 4:
    print(f"Warning: found {len(c_values)} distinct c values instead of 4: {c_values}")


def compute_performance_ratios(data, algorithms):
    values = (
        data.groupby(instance_columns + ["algorithm"], as_index=False)
        .agg(metric=(profile_metric, profile_agg))
    )

    metric_matrix = values.pivot(
        index=instance_columns,
        columns="algorithm",
        values="metric",
    ).reindex(columns=algorithms)

    valid = np.isfinite(metric_matrix) & (metric_matrix > 0)
    metric_matrix = metric_matrix.where(valid, np.inf)

    best_per_instance = metric_matrix.min(axis=1)
    solvable = np.isfinite(best_per_instance)
    metric_matrix = metric_matrix.loc[solvable]
    best_per_instance = best_per_instance.loc[solvable]

    if metric_matrix.empty:
        return metric_matrix

    return metric_matrix.div(best_per_instance, axis=0)


def choose_tau_max(ratio_matrices):
    finite_ratios = []
    for ratios in ratio_matrices:
        if ratios.empty:
            continue
        arr = ratios.to_numpy(dtype=float)
        finite_ratios.extend(arr[np.isfinite(arr)].tolist())

    finite_ratios = np.asarray(finite_ratios, dtype=float)
    finite_ratios = finite_ratios[finite_ratios >= 1.0]

    if finite_ratios.size == 0:
        return 2.0

    upper = np.quantile(finite_ratios, 0.98)
    return max(2.0, min(float(np.ceil(upper)), 100.0))


def plot_performance_profile(ax, ratios, tau_max):
    if ratios.empty:
        ax.text(
            0.5,
            0.5,
            "No valid instances",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    tau_grid = np.geomspace(1.0, tau_max, 400)
    n_instances = len(ratios)

    for alg in profile_algorithms:
        alg_ratios = ratios[alg].to_numpy(dtype=float)
        rho = np.array([np.count_nonzero(alg_ratios <= tau) / n_instances for tau in tau_grid])

        ax.step(
            tau_grid,
            rho,
            where="post",
            linewidth=1.6,
            linestyle=ALGO_LINESTYLES.get(alg, "-"),
            color=ALGO_COLORS.get(alg),
            label=algo_label(alg),
        )

    ax.set_xscale("log")
    ax.set_xlim(1.0, tau_max)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel(r"Performance ratio $\tau$")
    ax.set_ylabel(r"Fraction $\rho_s(\tau)$")
    ax.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


global_ratios = compute_performance_ratios(profile_source, profile_algorithms)
ratios_by_c = {
    c: compute_performance_ratios(profile_source[profile_source["c"] == c], profile_algorithms)
    for c in c_values
}

all_ratio_matrices = [global_ratios] + list(ratios_by_c.values())
tau_max = float(tau_max_override) if tau_max_override is not None else choose_tau_max(all_ratio_matrices)

fig, ax = plt.subplots(figsize=(7, 5.5), dpi=130)
plot_performance_profile(ax, global_ratios, tau_max)
ax.legend(loc="lower right", frameon=True)

fig.tight_layout()
if save:
    fig.savefig(
        figures_dir / f"performance_profile_global_{profile_metric.lower()}.png",
        dpi=220,
        bbox_inches="tight",
    )
plt.show()

for c in c_values:
    fig, ax = plt.subplots(figsize=(7, 5.5), dpi=130)
    plot_performance_profile(ax, ratios_by_c[c], tau_max)
    ax.legend(loc="lower right", frameon=True)

    fig.tight_layout()
    if save:
        c_filename = str(c).replace(".", "_")
        fig.savefig(
            figures_dir / f"performance_profile_c_{c_filename}_{profile_metric.lower()}.png",
            dpi=220,
            bbox_inches="tight",
        )
    plt.show()